In [1]:
!pip install -q kagglehub

In [2]:
import os, sys, json, platform
import numpy as np
import pandas as pd
import sklearn, joblib, kagglehub

In [13]:
import os, sys, json, platform
import numpy as np
import pandas as pd
import sklearn, joblib, kagglehub

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report,confusion_matrix

In [14]:
path = kagglehub.dataset_download("iammustafatz/diabetes-prediction-dataset")
print(path, os.listdir(path))
df = pd.read_csv(os.path.join(path, "diabetes_prediction_dataset.csv"))
display(df.head())
# If this fails, click Download on the Kaggle page, unzip the file, and upload the CSV with Colab's file panel. Then run df = pd.read_csv("diabetes_prediction_dataset.csv").

/root/.cache/kagglehub/datasets/iammustafatz/diabetes-prediction-dataset/versions/1 ['diabetes_prediction_dataset.csv']


,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,diabetes
0,Female,80.0,0,1,never,25.19,6.6,140,0
1,Female,54.0,0,0,No Info,27.32,6.6,80,0
2,Male,28.0,0,0,never,27.32,5.7,158,0
3,Female,36.0,0,0,current,23.45,5.0,155,0
4,Male,76.0,1,1,current,20.14,4.8,155,0


In [15]:
print(df.shape)
df.info()
display(df.describe())

(100000, 9)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 9 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   gender               100000 non-null  object 
 1   age                  100000 non-null  float64
 2   hypertension         100000 non-null  int64  
 3   heart_disease        100000 non-null  int64  
 4   smoking_history      100000 non-null  object 
 5   bmi                  100000 non-null  float64
 6   HbA1c_level          100000 non-null  float64
 7   blood_glucose_level  100000 non-null  int64  
 8   diabetes             100000 non-null  int64  
dtypes: float64(3), int64(4), object(2)
memory usage: 6.9+ MB


,age,hypertension,heart_disease,bmi,HbA1c_level,blood_glucose_level,diabetes
count,100000.000000,100000.00000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000
mean,41.885856,0.07485,0.039420,27.320767,5.527507,138.058060,0.085000
std,22.516840,0.26315,0.194593,6.636783,1.070672,40.708136,0.278883
min,0.080000,0.00000,0.000000,10.010000,3.500000,80.000000,0.000000
25%,24.000000,0.00000,0.000000,23.630000,4.800000,100.000000,0.000000
50%,43.000000,0.00000,0.000000,27.320000,5.800000,140.000000,0.000000
75%,60.000000,0.00000,0.000000,29.580000,6.200000,159.000000,0.000000
max,80.000000,1.00000,1.000000,95.690000,9.000000,300.000000,1.000000


In [16]:
print("\nTarget distribution:\n", df["diabetes"].value_counts(normalize=True))
print("\nGender:\n", df["gender"].value_counts())
print("\nSmoking history:\n", df["smoking_history"].value_counts())
print("\nMissing values:\n", df.isna().sum())

print("\nDuplicate rows:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
print("Shape after dropping duplicates:", df.shape)


Target distribution:
 diabetes
0    0.915
1    0.085
Name: proportion, dtype: float64

Gender:
 gender
Female    58552
Male      41430
Other        18
Name: count, dtype: int64

Smoking history:
 smoking_history
No Info        35816
never          35095
former          9352
current         9286
not current     6447
ever            4004
Name: count, dtype: int64

Missing values:
 gender                 0
age                    0
hypertension           0
heart_disease          0
smoking_history        0
bmi                    0
HbA1c_level            0
blood_glucose_level    0
diabetes               0
dtype: int64

Duplicate rows: 3854
Shape after dropping duplicates: (96146, 9)


In [17]:
CATEGORICAL = ["gender", "smoking_history"]
NUMERIC = ["age", "bmi", "HbA1c_level", "blood_glucose_level"]
BINARY = ["hypertension", "heart_disease"]
FEATURES = ["gender", "age", "hypertension", "heart_disease",
            "smoking_history", "bmi", "HbA1c_level", "blood_glucose_level"]
TARGET = "diabetes"
RANDOM_STATE = 42

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print("Train:", X_train.shape, " Test:", X_test.shape)

Train: (76916, 8)  Test: (19230, 8)


In [18]:
preprocess = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL),
    ("num", StandardScaler(), NUMERIC),
    ("bin", "passthrough", BINARY),
])

pipe = Pipeline([
    ("prep", preprocess),
    ("clf", LogisticRegression()),   # placeholder; the grid swaps it out
])

param_grid = [
    {"clf": [LogisticRegression(max_iter=1000)],
     "clf__C": [0.1, 1, 10],
     "clf__class_weight": [None, "balanced"]},
    {"clf": [RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)],
     "clf__max_depth": [8, 12],
     "clf__min_samples_leaf": [1, 5]},
    {"clf": [HistGradientBoostingClassifier(random_state=RANDOM_STATE)],
     "clf__learning_rate": [0.05, 0.1],
     "clf__max_depth": [None, 6]},
]

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
grid = GridSearchCV(pipe, param_grid, cv=cv, scoring="roc_auc", n_jobs=-1, verbose=1)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
print("Best CV ROC-AUC:", round(grid.best_score_, 4))
print("Best model     :", type(best_model.named_steps["clf"]).__name__)
print("Best params    :", grid.best_params_)

Fitting 3 folds for each of 14 candidates, totalling 42 fits
Best CV ROC-AUC: 0.9791
Best model     : HistGradientBoostingClassifier
Best params    : {'clf': HistGradientBoostingClassifier(random_state=42), 'clf__learning_rate': 0.1, 'clf__max_depth': 6}


In [19]:
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

metrics = {
    "accuracy":  round(accuracy_score(y_test, y_pred), 4),
    "precision": round(precision_score(y_test, y_pred), 4),
    "recall":    round(recall_score(y_test, y_pred), 4),
    "f1":        round(f1_score(y_test, y_pred), 4),
    "roc_auc":   round(roc_auc_score(y_test, y_proba), 4),
    "cv_roc_auc": round(grid.best_score_, 4),
}
print(json.dumps(metrics, indent=2))
print(classification_report(y_test, y_pred))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))

{
  "accuracy": 0.9718,
  "precision": 0.984,
  "recall": 0.691,
  "f1": 0.8119,
  "roc_auc": 0.9778,
  "cv_roc_auc": 0.9791
}
              precision    recall  f1-score   support

           0       0.97      1.00      0.98     17534
           1       0.98      0.69      0.81      1696

    accuracy                           0.97     19230
   macro avg       0.98      0.84      0.90     19230
weighted avg       0.97      0.97      0.97     19230

Confusion matrix:
 [[17515    19]
 [  524  1172]]


In [20]:
os.makedirs("artifacts", exist_ok=True)

joblib.dump(best_model, "artifacts/diabetes_model.joblib")

def to_json_safe(v):
    return v if isinstance(v, (int, float, str, bool, type(None))) else str(v)

model_params = {
    "model_name": "diabetes-classifier",
    "model_type": type(best_model.named_steps["clf"]).__name__,
    "best_params": {k: to_json_safe(v) for k, v in grid.best_params_.items()},
    "features": FEATURES,
    "categorical_features": CATEGORICAL,
    "numeric_features": NUMERIC,
    "binary_features": BINARY,
    "categories": {c: sorted(df[c].unique().tolist()) for c in CATEGORICAL},
    "target": TARGET,
    "random_state": RANDOM_STATE,
    "train_size": len(X_train),
    "test_size": len(X_test),
    "python_version": platform.python_version(),
    "sklearn_version": sklearn.__version__,
}
with open("artifacts/model_params.json", "w") as f:
    json.dump(model_params, f, indent=2)

with open("artifacts/metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

test_df = X_test.copy()
test_df[TARGET] = y_test
test_df.to_csv("artifacts/test_data.csv", index=False)

print(os.listdir("artifacts"))

['model_params.json', 'test_data.csv', 'metrics.json', 'diabetes_model.joblib']


In [22]:
loaded = joblib.load("artifacts/diabetes_model.joblib")
sample = pd.DataFrame([{"gender": "Female", "age": 54, "hypertension": 0,
                        "heart_disease": 0, "smoking_history": "former",
                        "bmi": 32.5, "HbA1c_level": 7.2,
                        "blood_glucose_level": 200}])
print("Prediction:", loaded.predict(sample)[0],
      " Probability:", round(loaded.predict_proba(sample)[0][1], 4))

# Round to 4 decimal places to match the stored metrics["accuracy"]
loaded_acc = round(accuracy_score(y_test, loaded.predict(X_test)), 4)
assert loaded_acc == metrics["accuracy"]
print("Reloaded model matches (accuracy: {})".format(loaded_acc))

Prediction: 1  Probability: 0.9996
Reloaded model matches (accuracy: 0.9718)


In [23]:
# Download the file
!zip -r artifacts.zip artifacts
from google.colab import files
files.download("artifacts.zip")

  adding: artifacts/ (stored 0%)
  adding: artifacts/model_params.json (deflated 60%)
  adding: artifacts/test_data.csv (deflated 81%)
  adding: artifacts/metrics.json (deflated 34%)
  adding: artifacts/diabetes_model.joblib (deflated 71%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>